# Childhood Pneumonia Prediction — Comparative ML Framework

**Dataset:** BD_IRA.xlsx (801 pediatric patients, Children's Hospital of Rabat, Morocco)

This notebook implements the full corrected machine learning pipeline:
- 28 engineered features (clinical, demographic, socio-environmental, vaccination)
- Leakage-free preprocessing (StandardScaler fit on train only)
- SMOTE class imbalance correction (applied within CV folds only)
- 11 supervised classification algorithms, including XGBoost and LightGBM
- Hyperparameter tuning (RandomizedSearchCV) for top 3 models
- Statistical validation (Mann-Whitney U, chi-square)
- Feature importance (Random Forest) + SHAP interpretability
- Publication-ready figures (ROC curves, confusion matrices, SHAP summary, etc.)


## 1. Setup & Imports

In [ ]:
!pip install xgboost lightgbm shap imbalanced-learn scikit-learn matplotlib seaborn openpyxl -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import json, os

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                      RandomizedSearchCV, train_test_split)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve,
                              confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy.stats import randint, uniform, mannwhitneyu, chi2_contingency

os.makedirs('figs', exist_ok=True)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10})
print("Libraries loaded successfully.")

## 2. Load Data & Engineer Features

We load the raw clinical dataset and construct the target variable plus 28 predictive
features spanning four categories: **Clinical biomarkers**, **Demographic variables**,
**Socio-environmental determinants**, and **Vaccination history**.

In [ ]:
df = pd.read_excel('BD_IRA.xlsx')  # update path if needed
print("Raw shape:", df.shape)
df.head()

In [ ]:
# ── Target variable: pneumonia-positive diagnoses ──────────────────────────
pneumonia_diags = ['Pneumonia', 'Dyspneic pneumopathy', 'Staphylococcal pneumonia',
                   'Bronchiolitis + Pneumonia', 'Asthma attack + Pneumonia',
                   'Pleuropneumonitis', 'Interstitial pneumopathy']
df['target'] = df['Main diagnostic '].isin(pneumonia_diags).astype(int)

print(df['target'].value_counts())
print(f"Positive (pneumonia) rate: {df['target'].mean():.1%}")

In [ ]:
# ── Helper: convert Yes/No columns to binary ────────────────────────────────
def yn(col):
    return (df[col].str.strip().str.lower() == 'yes').astype(int)

feat = pd.DataFrame()

# --- Clinical biomarkers ---
feat['CRP']             = pd.to_numeric(df['C-reactive protein'], errors='coerce')
feat['Procalcitonin']   = pd.to_numeric(df['Procalcitonin'], errors='coerce')
feat['SaO2']            = pd.to_numeric(df['Oxygen saturation (SaO2) at admission'], errors='coerce')
feat['Temperature']     = pd.to_numeric(df['Axillary temperature (\u00b0C)'], errors='coerce')
feat['RespiratoryRate'] = pd.to_numeric(df['Respiratory rate'], errors='coerce')
feat['HeartRate']       = pd.to_numeric(df['Heart rate'], errors='coerce')

# --- Clinical signs ---
feat['Fever']        = yn('History of fever')
feat['Crackles']     = yn('Crackles')
feat['Wheezing']     = yn('Wheezing')
feat['NasalFlaring'] = yn('Nasal flaring')
feat['Cyanosis']     = yn('Cyanosis')

# --- Demographic ---
feat['Age']          = df['Age (months)'].astype(float)
feat['Gender']       = (df['Gender '].str.strip() == 'Male').astype(int)
feat['Siblings']     = pd.to_numeric(df['Number of siblings'], errors='coerce')
feat['Prematurity']  = yn('Health history : Prematurity                      ')
feat['Breastfed']    = yn('Breastfeeding')
feat['PriorRespAdm'] = yn('Health history : Prior admission because of respiratory condition')

# --- Socio-environmental (KEY research variables) ---
feat['HouseholdSize']  = pd.to_numeric(df['Number of persons living in house'], errors='coerce')
feat['Rooms']          = pd.to_numeric(df['Number of rooms in house'], errors='coerce')
feat['PersonsPerRoom'] = feat['HouseholdSize'] / feat['Rooms'].replace(0, np.nan)
feat['SmokersAtHome']  = yn('Smokers at home')
feat['VitaminA']       = yn('Vitamin A supplementation ')

# --- Vaccination (one-hot + dose counts) ---
vax = df['Age-appropriate vaccinations'].str.strip()
feat['Vacc_Full']    = (vax == 'Yes').astype(int)
feat['Vacc_Partial'] = (vax == 'Partially').astype(int)
feat['Vacc_None']    = (vax == 'No').astype(int)
feat['HibDoses']     = pd.to_numeric(df['Number of Hib vaccine doses received'], errors='coerce')
feat['PneumoDoses']  = pd.to_numeric(df['Number of pneumococcal vaccine doses received'], errors='coerce')

# --- Chest X-ray (ordinal encode) ---
xray_map = {'Normal': 0, 'NF': 1, 'Other infiltrates': 2, 'Endpoint pneumonia': 3}
feat['ChestXray'] = df['Chest X-ray finding'].map(xray_map)

feat['target'] = df['target'].values

print(f"Engineered {feat.shape[1]-1} features.")
feat.head()

In [ ]:
# ── Clean: drop rows missing critical fields, impute remaining with median ──
feat = feat.dropna(subset=['CRP', 'Age', 'RespiratoryRate', 'HouseholdSize', 'target'])
feat = feat.fillna(feat.median(numeric_only=True))

FEATURE_COLS = [c for c in feat.columns if c != 'target']
X = feat[FEATURE_COLS].values
y = feat['target'].values

print(f"Final dataset: {feat.shape[0]} rows, {len(FEATURE_COLS)} features")
print(f"Class distribution: {dict(feat['target'].value_counts())}")
print(f"Positive rate: {feat['target'].mean():.3f}")

## 3. Train-Test Split (Leakage-Free)

The `StandardScaler` is fit **only** on training data. This avoids the data-leakage
issue present in the original baseline pipeline.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit ONLY on train
X_test_sc  = scaler.transform(X_test)        # transform test using train statistics

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
print(f"Train positive rate: {y_train.mean():.3f} | Test positive rate: {y_test.mean():.3f}")

## 4. Hyperparameter Tuning (Top 3 Models)

`RandomizedSearchCV` is used to tune XGBoost, LightGBM, and Logistic Regression.
SMOTE is embedded inside the pipeline so oversampling happens **within each CV
training fold only** — never on validation/test data.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def tune(name, est, param_dist, X_tr, y_tr, n_iter=30):
    pipe = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('scaler', StandardScaler()),
        ('clf', est)
    ])
    param_dist_prefixed = {'clf__' + k: v for k, v in param_dist.items()}
    rs = RandomizedSearchCV(pipe, param_dist_prefixed, n_iter=n_iter,
                            scoring='roc_auc', cv=cv, random_state=42, n_jobs=-1)
    rs.fit(X_tr, y_tr)
    print(f"{name}: best CV-AUC = {rs.best_score_:.4f}")
    print(f"  Best params: {rs.best_params_}\n")
    return rs.best_estimator_, rs.best_score_

In [ ]:
# --- XGBoost tuning ---
xgb_params = {
    'n_estimators': randint(100, 400),
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'scale_pos_weight': [1, 2, 3],
}
xgb_best, xgb_cv = tune('XGBoost', XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
                          xgb_params, X_train, y_train, n_iter=40)

In [ ]:
# --- LightGBM tuning ---
lgb_params = {
    'n_estimators': randint(100, 400),
    'num_leaves': randint(20, 80),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_samples': randint(5, 30),
}
lgb_best, lgb_cv = tune('LightGBM', LGBMClassifier(random_state=42, verbose=-1),
                          lgb_params, X_train, y_train, n_iter=40)

In [ ]:
# --- Logistic Regression tuning ---
lr_params = {
    'C': uniform(0.01, 10),
    'solver': ['lbfgs', 'saga'],
    'max_iter': [500, 1000],
}
lr_best, lr_cv = tune('Logistic Regression', LogisticRegression(random_state=42),
                       lr_params, X_train, y_train, n_iter=20)

## 5. Evaluate All 11 Models

Each model is wrapped in a SMOTE + StandardScaler pipeline. Cross-validated AUC
(5-fold stratified) is the primary selection metric; final metrics are reported
on the untouched held-out test set.

In [ ]:
def make_pipe(clf):
    return ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('scaler', StandardScaler()),
        ('clf', clf)
    ])

models = {
    'Logistic Regression': lr_best,
    'Decision Tree':       make_pipe(DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)),
    'Random Forest':       make_pipe(RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)),
    'Gradient Boosting':   make_pipe(GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42)),
    'AdaBoost':            make_pipe(AdaBoostClassifier(n_estimators=100, random_state=42)),
    'XGBoost (Tuned)':     xgb_best,
    'LightGBM (Tuned)':    lgb_best,
    'SVM (RBF)':           make_pipe(SVC(kernel='rbf', probability=True, random_state=42)),
    'KNN (k=7)':           make_pipe(KNeighborsClassifier(n_neighbors=7)),
    'Naive Bayes':         make_pipe(GaussianNB()),
    'MLP (100,50)':        make_pipe(MLPClassifier(hidden_layer_sizes=(100,50), max_iter=500, random_state=42)),
}
print(f"{len(models)} models configured.")

In [ ]:
results = {}
cv_scores_all = {}

for name, pipe in models.items():
    # Cross-validated AUC (5-fold stratified, SMOTE applied within folds)
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_scores_all[name] = cv_auc

    # Fit on full training set, evaluate on held-out test set
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'AUC-ROC':   roc_auc_score(y_test, y_prob),
        'CV-AUC Mean': cv_auc.mean(),
        'CV-AUC Std':  cv_auc.std(),
        'y_prob': y_prob,
        'y_pred': y_pred,
    }
    print(f"{name:25s}  CV-AUC={cv_auc.mean():.4f}\u00b1{cv_auc.std():.4f}  "
          f"Test-AUC={roc_auc_score(y_test, y_prob):.4f}  Recall={recall_score(y_test, y_pred):.3f}")

In [ ]:
# ── Results summary table ───────────────────────────────────────────────────
summary = pd.DataFrame({
    name: {k: v for k, v in r.items() if k not in ('y_prob', 'y_pred')}
    for name, r in results.items()
}).T.sort_values('CV-AUC Mean', ascending=False)

summary.round(4)

## 6. Statistical Validation (Mann-Whitney U & Chi-Square)

In [ ]:
print("=== Mann-Whitney U tests (continuous features) ===")
for col in ['CRP', 'Procalcitonin', 'Age', 'RespiratoryRate', 'SaO2', 'Temperature']:
    g0 = feat.loc[feat['target']==0, col].dropna()
    g1 = feat.loc[feat['target']==1, col].dropna()
    stat, p = mannwhitneyu(g1, g0, alternative='two-sided')
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'NS'))
    print(f"  {col:20s}  U={stat:.0f}  p={p:.4f} {sig}")

In [ ]:
print("=== Chi-square tests (categorical features) ===")
for col in ['SmokersAtHome', 'Fever', 'Vacc_Full', 'Crackles', 'NasalFlaring', 'Wheezing', 'Breastfed']:
    ct = pd.crosstab(feat[col], feat['target'])
    chi2, p, dof, _ = chi2_contingency(ct)
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'NS'))
    print(f"  {col:20s}  chi2={chi2:.3f} df={dof}  p={p:.4f} {sig}")

# Household crowding (quartiles of persons-per-room)
ct2 = pd.crosstab(
    pd.qcut(feat['PersonsPerRoom'].dropna(), q=4, duplicates='drop'),
    feat.loc[feat['PersonsPerRoom'].notna(), 'target']
)
chi2_c, p_c, dof_c, _ = chi2_contingency(ct2)
sig_c = '***' if p_c<0.001 else ('**' if p_c<0.01 else ('*' if p_c<0.05 else 'NS'))
print(f"  {'PersonsPerRoom':20s}  chi2={chi2_c:.3f} df={dof_c}  p={p_c:.4f} {sig_c}")

In [ ]:
# ── Vaccination-status prevalence gradient ─────────────────────────────────
vax_prev = {}
for label, mask in [('Full', feat['Vacc_Full']==1),
                    ('Partial', feat['Vacc_Partial']==1),
                    ('None', feat['Vacc_None']==1)]:
    prev = feat.loc[mask, 'target'].mean()
    n = mask.sum()
    vax_prev[label] = (prev, n)
    print(f"Pneumonia prevalence [{label:7s}] (n={n:3d}): {prev:.1%}")

## 7. Feature Importance (Random Forest)

In [ ]:
rf_pipe = make_pipe(RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1))
rf_pipe.fit(X_train, y_train)
rf_clf = rf_pipe.named_steps['clf']

feat_imp = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Importance': rf_clf.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

feat_imp.head(15)

## 8. SHAP Interpretability (XGBoost)

In [ ]:
xgb_pipe = xgb_best
xgb_pipe.fit(X_train, y_train)

# Transform test data through preprocessing steps (excluding final classifier)
Xtest_transformed = X_test.copy()
for step_name, step in xgb_pipe.steps[:-1]:
    if hasattr(step, 'transform'):
        try:
            Xtest_transformed = step.transform(Xtest_transformed)
        except Exception:
            pass

xgb_clf = xgb_pipe.named_steps['clf']
explainer = shap.TreeExplainer(xgb_clf)
shap_vals = explainer.shap_values(Xtest_transformed)
print("SHAP values computed:", shap_vals.shape)

## 9. Figures

Publication-ready figures: ROC curves, CV-AUC with confidence intervals, feature
importance, confusion matrices, SHAP summary, vaccination prevalence, and
household-crowding analysis.

In [ ]:
BLUE, GOLD, RED, GREEN = '#1a3a5c', '#e8a020', '#c0392b', '#27ae60'
COLORS = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
          '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf','#aec7e8']

In [ ]:
# ── Fig 1: ROC Curves ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0,1],[0,1],'k--', alpha=0.4, lw=1, label='Random (AUC=0.500)')
for (name, res), col in zip(results.items(), COLORS):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=col, lw=1.8, label=f"{name} (AUC={res['AUC-ROC']:.3f})")
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves \u2014 All Models (SMOTE + Corrected Pipeline)', fontsize=13, fontweight='bold', color=BLUE)
ax.legend(loc='lower right', fontsize=8.5, framealpha=0.9)
plt.tight_layout()
plt.savefig('figs/fig1_roc_curves.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 2: CV-AUC with 95% confidence intervals ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
names_sorted = sorted(results.keys(), key=lambda n: results[n]['CV-AUC Mean'], reverse=True)
means = [results[n]['CV-AUC Mean'] for n in names_sorted]
stds  = [results[n]['CV-AUC Std']  for n in names_sorted]
ci95  = [1.96*s for s in stds]
colors_bar = [GREEN if m == max(means) else ('#e74c3c' if '(Tuned)' in n else BLUE)
              for m, n in zip(means, names_sorted)]

ax.barh(range(len(names_sorted)), means, xerr=ci95, color=colors_bar, alpha=0.85,
        edgecolor='white', height=0.6,
        error_kw={'elinewidth':1.5, 'capsize':4, 'capthick':1.5, 'ecolor':'#333'})
ax.set_yticks(range(len(names_sorted)))
ax.set_yticklabels(names_sorted, fontsize=10)
ax.set_xlabel('Cross-Validated AUC-ROC (mean \u00b1 95% CI)', fontsize=11)
ax.set_title('Model Comparison \u2014 CV-AUC with 95% Confidence Intervals', fontsize=12, fontweight='bold', color=BLUE)
ax.axvline(0.5, color='red', lw=1, linestyle='--', alpha=0.5)
for i, (m, c) in enumerate(zip(means, ci95)):
    ax.text(m + c + 0.003, i, f'{m:.3f}\u00b1{c:.3f}', va='center', fontsize=8)
ax.set_xlim(0.4, 1.02)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('figs/fig2_cv_auc_ci.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 3: Feature Importance (top 15, color-coded by category) ────────────
cats = {'CRP':'Clinical','Procalcitonin':'Clinical','SaO2':'Clinical',
        'Temperature':'Clinical','RespiratoryRate':'Clinical','HeartRate':'Clinical',
        'Fever':'Clinical','Crackles':'Clinical','Wheezing':'Clinical',
        'NasalFlaring':'Clinical','Cyanosis':'Clinical', 'ChestXray':'Clinical',
        'Age':'Demographic','Gender':'Demographic','Siblings':'Demographic',
        'Prematurity':'Demographic','Breastfed':'Demographic','PriorRespAdm':'Demographic',
        'HouseholdSize':'Socio-Env','Rooms':'Socio-Env','PersonsPerRoom':'Socio-Env',
        'SmokersAtHome':'Socio-Env','VitaminA':'Socio-Env',
        'Vacc_Full':'Vaccination','Vacc_Partial':'Vaccination','Vacc_None':'Vaccination',
        'HibDoses':'Vaccination','PneumoDoses':'Vaccination'}
cat_colors = {'Clinical':'#2980b9','Demographic':'#8e44ad','Socio-Env':'#27ae60','Vaccination':'#e67e22'}

top_n = 15
fi_top = feat_imp.head(top_n)
bar_colors = [cat_colors.get(cats.get(f,'Clinical'),'#555') for f in fi_top['Feature']]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(fi_top)), fi_top['Importance'].values, color=bar_colors, edgecolor='white', height=0.65)
ax.set_yticks(range(len(fi_top)))
ax.set_yticklabels(fi_top['Feature'].values, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Gini Importance (Mean Decrease in Impurity)', fontsize=11)
ax.set_title(f'Top {top_n} Feature Importances \u2014 Random Forest (300 trees)', fontsize=12, fontweight='bold', color=BLUE)
patches = [mpatches.Patch(color=c, label=l) for l, c in cat_colors.items()]
ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('figs/fig3_feature_importance.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 4: Confusion matrices (top 3 models) ────────────────────────────────
top3 = sorted(results.keys(), key=lambda n: results[n]['CV-AUC Mean'], reverse=True)[:3]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name in zip(axes, top3):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-Pneumonia','Pneumonia'],
                yticklabels=['Non-Pneumonia','Pneumonia'],
                annot_kws={'size':13, 'weight':'bold'})
    ax.set_title(f'{name}\nAUC={results[name]["AUC-ROC"]:.3f}  F1={results[name]["F1"]:.3f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)
fig.suptitle('Confusion Matrices \u2014 Top 3 Models', fontsize=13, fontweight='bold', color=BLUE, y=1.05)
plt.tight_layout()
plt.savefig('figs/fig4_confusion_matrices.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 5: SHAP Summary Plot ────────────────────────────────────────────────
shap.summary_plot(shap_vals, Xtest_transformed, feature_names=FEATURE_COLS,
                  max_display=15, show=False, plot_size=(10, 8))
plt.title('SHAP Summary Plot \u2014 XGBoost (Tuned)\nFeature Impact on Pneumonia Prediction',
          fontsize=12, fontweight='bold', color=BLUE, pad=15)
plt.tight_layout()
plt.savefig('figs/fig5_shap_summary.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 6: Pneumonia prevalence by vaccination status ──────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
labels = [f"Fully\nVaccinated\n(n={vax_prev['Full'][1]})",
          f"Partially\nVaccinated\n(n={vax_prev['Partial'][1]})",
          f"Unvaccinated\n(n={vax_prev['None'][1]})"]
prevs  = [vax_prev['Full'][0]*100, vax_prev['Partial'][0]*100, vax_prev['None'][0]*100]
bars = ax.bar(labels, prevs, color=[GREEN, GOLD, RED], edgecolor='white', width=0.5)
for bar, val in zip(bars, prevs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.8,
            f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Pneumonia Prevalence (%)', fontsize=11)
ax.set_title('Pneumonia Prevalence by Vaccination Status', fontsize=12, fontweight='bold', color=BLUE)
ax.set_ylim(0, 65)
ax.axhline(feat['target'].mean()*100, color='grey', lw=1.5, linestyle='--', alpha=0.7,
           label=f"Overall prevalence ({feat['target'].mean()*100:.1f}%)")
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figs/fig6_vaccination_prevalence.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# ── Fig 7: Household crowding analysis ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
data0 = feat.loc[feat['target']==0, 'PersonsPerRoom'].dropna()
data1 = feat.loc[feat['target']==1, 'PersonsPerRoom'].dropna()
bp = ax.boxplot([data0, data1], patch_artist=True, medianprops={'color':'black','linewidth':2})
bp['boxes'][0].set_facecolor('#3498db')
bp['boxes'][1].set_facecolor('#e74c3c')
ax.set_xticklabels(['Non-Pneumonia', 'Pneumonia'], fontsize=11)
ax.set_ylabel('Persons per Room', fontsize=11)
ax.set_title('Household Crowding by Pneumonia Status', fontsize=11, fontweight='bold', color=BLUE)

ax2 = axes[1]
ax2.hist(data0, bins=20, alpha=0.6, color='#3498db', label='Non-Pneumonia', density=True)
ax2.hist(data1, bins=20, alpha=0.6, color='#e74c3c', label='Pneumonia', density=True)
ax2.set_xlabel('Persons per Room', fontsize=11)
ax2.set_ylabel('Density', fontsize=11)
ax2.set_title('Distribution by Pneumonia Status', fontsize=11, fontweight='bold', color=BLUE)
ax2.legend(fontsize=10)

fig.suptitle('Household Crowding as a Socio-Environmental Predictor', fontsize=13, fontweight='bold', color=BLUE)
plt.tight_layout()
plt.savefig('figs/fig7_crowding_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

## 10. Save Results

In [ ]:
results_json = {k: {m: v for m, v in r.items() if m not in ('y_prob','y_pred')}
                for k, r in results.items()}

output = {
    'dataset': {
        'n_total': int(feat.shape[0]),
        'n_features': len(FEATURE_COLS),
        'n_pos': int(feat['target'].sum()),
        'n_neg': int((feat['target']==0).sum()),
        'pos_rate': float(feat['target'].mean()),
    },
    'models': results_json,
    'feature_importance': feat_imp.head(15).to_dict('records'),
    'cv_scores': {k: {'mean': float(v.mean()), 'std': float(v.std()),
                       'ci95': float(1.96*v.std())} for k, v in cv_scores_all.items()},
    'vaccination_prevalence': {k: {'prev': float(v[0]), 'n': int(v[1])} for k, v in vax_prev.items()},
    'best_model': max(results_json.keys(), key=lambda n: results_json[n]['CV-AUC Mean']),
    'top3': sorted(results_json.keys(), key=lambda n: results_json[n]['CV-AUC Mean'], reverse=True)[:3],
}

with open('ml_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("=== SUMMARY ===")
print(f"Best model: {output['best_model']}")
print(f"Best CV-AUC: {output['cv_scores'][output['best_model']]['mean']:.4f} "
      f"\u00b1 {output['cv_scores'][output['best_model']]['ci95']:.4f}")
print("\nAll figures saved to figs/")
print("Results saved to ml_results.json")